# Case Study 02 — EDA: Macro-Credit Transmission

**Erick Condoy** · Economist (UNL) · Quant Researcher  
Repository: [credit-risk-lab](https://github.com/EryckFCS/credit-risk-lab)

---

## Objective

Understand the **data-generating process** before modeling.
Three analytical axes:

| # | Axis | Key Question |
|---|------|-------------|
| 1 | Temporal distribution of target | Is deterioration idiosyncratic or systemic? |
| 2 | Feature-target correlations | Which features carry signal? Multicollinearity risk? |
| 3 | Macro-credit transmission | How many periods does macro take to impact NPL? |

Findings from this notebook directly justify modeling decisions in NB04.

In [ ]:
import warnings
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

BASE  = Path('../')
PANEL = BASE / 'data' / 'panel'
FIGS  = BASE / 'reports' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

# Institutional palette
TEAL   = '#01696f'; MAROON = '#a12c7b'; GRAY  = '#bab9b4'
GREEN  = '#437a22'; ORANGE = '#964219'; GOLD  = '#d19900'
BLUE   = '#006494'; PURPLE = '#7a39bb'
PAL6   = [TEAL, ORANGE, BLUE, GREEN, MAROON, GOLD]

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False,    'axes.spines.right': False,
    'axes.edgecolor': '#dcd9d5', 'axes.labelcolor': '#28251d',
    'xtick.color': '#7a7974',    'ytick.color': '#7a7974',
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.titlesize': 11,        'axes.titleweight': 'semibold',
    'figure.dpi': 130,
})
print('Environment ready.')

## 1. Load Panel

In [ ]:
df = pd.read_parquet(PANEL / 'panel_model_ready.parquet')
with open(PANEL / 'panel_metadata.json') as f:
    meta = json.load(f)

ALL_FEATURES  = meta['all_features']
SUPV_FEATURES = meta['supervisory_features']
MACRO_FEATURES= meta['macro_features']
TARGET        = meta['target']

df['period'] = pd.to_datetime(df['period'])

print(f'Panel: {len(df):,} rows | {df.institution_id.nunique()} institutions | {df.period.nunique()} periods')
print(f'Target rate: {df[TARGET].mean():.2%}')
print(f'Period: {df.period.min().date()} — {df.period.max().date()}')

## Axis 1 — Temporal Distribution of Target

In [ ]:
# --- 1a. Deterioration rate per period ---
period_rate = df.groupby('period')[TARGET].mean().reset_index()
period_rate.columns = ['period', 'deterioration_rate']

# --- 1b. NPL system average per period ---
if 'npl_ratio' in df.columns:
    npl_avg = df.groupby('period')['npl_ratio'].mean().reset_index()

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# Plot 1: Deterioration rate over time
axes[0].bar(period_rate['period'], period_rate['deterioration_rate'],
            color=TEAL, alpha=0.75, width=25)
axes[0].axhline(period_rate['deterioration_rate'].mean(),
                color=ORANGE, ls='--', lw=1.2, label=f'Mean: {period_rate["deterioration_rate"].mean():.2%}')
axes[0].set_ylabel('Deterioration Rate')
axes[0].set_title('Target: Deterioration Rate per Period (y_deterioration)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[0].legend(frameon=False, fontsize=9)

# Plot 2: NPL system average
if 'npl_ratio' in df.columns:
    axes[1].plot(npl_avg['period'], npl_avg['npl_ratio'],
                 color=MAROON, lw=2, marker='o', markersize=3)
    axes[1].fill_between(npl_avg['period'], npl_avg['npl_ratio'],
                          alpha=0.12, color=MAROON)
    axes[1].set_ylabel('NPL Ratio (system avg)')
    axes[1].set_title('System-Wide NPL Ratio Over Time')
    axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))

plt.tight_layout(h_pad=1.5)
plt.savefig(FIGS / 'eda_01_temporal_target.png', bbox_inches='tight', dpi=150)
plt.show()

# Systemic vs idiosyncratic check
high_periods = period_rate[period_rate['deterioration_rate'] > 0.5]['period']
print(f'High-deterioration periods (>50% institutions): {len(high_periods)}')
if len(high_periods) > 0:
    print('  → Systemic episodes detected. Fixed effects or time dummies recommended.')
else:
    print('  → No dominant systemic episodes. Idiosyncratic pattern dominates.')

In [ ]:
# --- 1c. Heatmap: institution × period deterioration ---
heat = df.pivot_table(index='institution_id', columns='period', values=TARGET)
heat.columns = [str(c.date()) for c in heat.columns]

fig, ax = plt.subplots(figsize=(max(10, len(heat.columns)*0.7), max(4, len(heat)*0.5)))
sns.heatmap(heat, cmap='RdYlGn_r', linewidths=0.4, linecolor='#f3f0ec',
            ax=ax, cbar_kws={'label': 'Deterioration (1=Yes)', 'shrink': 0.6},
            annot=len(heat.columns) <= 16, fmt='.0f', annot_kws={'size': 8})
ax.set_title('Deterioration Heatmap: Institution × Period', pad=12)
ax.set_xlabel('Period'); ax.set_ylabel('Institution')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(FIGS / 'eda_02_heatmap_deterioration.png', bbox_inches='tight', dpi=150)
plt.show()

## Axis 2 — Feature–Target Correlations

In [ ]:
from scipy.stats import pointbiserialr

corr_results = []
for feat in ALL_FEATURES:
    if feat not in df.columns:
        continue
    s = df[[feat, TARGET]].dropna()
    if len(s) < 10:
        continue
    r, p = pointbiserialr(s[TARGET], s[feat])
    corr_results.append({
        'feature': feat,
        'point_biserial_r': round(r, 4),
        'p_value': round(p, 4),
        'abs_r': abs(r),
        'signal': 'strong' if abs(r) > 0.25 else ('moderate' if abs(r) > 0.10 else 'weak'),
        'significant': p < 0.05
    })

corr_df = pd.DataFrame(corr_results).sort_values('abs_r', ascending=False)

# Visual
fig, ax = plt.subplots(figsize=(8, max(5, len(corr_df)*0.45)))
colors = [TEAL if r >= 0 else MAROON for r in corr_df['point_biserial_r']]
bars = ax.barh(corr_df['feature'], corr_df['point_biserial_r'], color=colors, alpha=0.8)
ax.axvline(0, color=GRAY, lw=0.8)
ax.axvline(0.10, color=GRAY, lw=0.6, ls=':')
ax.axvline(-0.10, color=GRAY, lw=0.6, ls=':')
ax.axvline(0.25, color=ORANGE, lw=0.8, ls='--', label='|r|=0.25 (strong)')
ax.axvline(-0.25, color=ORANGE, lw=0.8, ls='--')
ax.set_xlabel('Point-Biserial r with y_deterioration')
ax.set_title('Feature–Target Correlation (Point-Biserial)')
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / 'eda_03_feature_target_corr.png', bbox_inches='tight', dpi=150)
plt.show()

display(corr_df.drop(columns='abs_r'))

In [ ]:
# --- Multicollinearity check: correlation matrix of features ---
feat_cols = [f for f in ALL_FEATURES if f in df.columns]
corr_matrix = df[feat_cols].corr()

fig, ax = plt.subplots(figsize=(max(8, len(feat_cols)*0.7), max(7, len(feat_cols)*0.65)))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, annot_kws={'size': 7},
            linewidths=0.3, linecolor='#f3f0ec',
            cbar_kws={'shrink': 0.6})
ax.set_title('Feature Correlation Matrix (multicollinearity audit)', pad=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(FIGS / 'eda_04_feature_corr_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

# Flag high correlations
high_corr = []
for i in range(len(corr_matrix)):
    for j in range(i+1, len(corr_matrix)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.80:
            high_corr.append((corr_matrix.index[i], corr_matrix.columns[j], round(val, 3)))

if high_corr:
    print('HIGH COLLINEARITY (|r|>0.80) — consider dropping one of each pair:')
    for a, b, r in high_corr:
        print(f'  {a} × {b} = {r}')
else:
    print('No critical multicollinearity detected (all |r| < 0.80).')

## Axis 3 — Macro-Credit Transmission Analysis

In [ ]:
# Cross-correlation between macro variables and NPL at different lags
# Answers: how many periods does macro take to transmit to credit quality?

system_ts = df.groupby('period').agg(
    npl_ratio=('npl_ratio', 'mean'),
    activity_growth=('activity_growth_lag1', 'mean'),
    credit_system_growth=('credit_system_growth_lag1', 'mean'),
).dropna().sort_index()

macro_vars = [c for c in ['activity_growth', 'credit_system_growth'] if c in system_ts.columns]

if len(system_ts) >= 8 and len(macro_vars) >= 1:
    fig, axes = plt.subplots(1, len(macro_vars), figsize=(6*len(macro_vars), 4), squeeze=False)
    max_lag = min(8, len(system_ts) // 3)

    for idx, macro_var in enumerate(macro_vars):
        lags = range(-max_lag, max_lag+1)
        xcorrs = [system_ts['npl_ratio'].corr(system_ts[macro_var].shift(lag)) for lag in lags]
        colors = [TEAL if abs(x) == max(map(abs, xcorrs)) else (MAROON if x < 0 else BLUE) for x in xcorrs]
        axes[0][idx].bar(list(lags), xcorrs, color=colors, alpha=0.75)
        axes[0][idx].axhline(0, color=GRAY, lw=0.8)
        axes[0][idx].axhline(0.3,  color=ORANGE, lw=0.7, ls='--', alpha=0.6)
        axes[0][idx].axhline(-0.3, color=ORANGE, lw=0.7, ls='--', alpha=0.6)
        axes[0][idx].set_xlabel('Lag (periods, + = macro leads)')
        axes[0][idx].set_ylabel('Cross-correlation with NPL')
        axes[0][idx].set_title(f'NPL vs {macro_var}\nCross-Correlation by Lag')
        best_lag = list(lags)[xcorrs.index(max(xcorrs, key=abs))]
        axes[0][idx].annotate(f'Peak lag={best_lag}', xy=(best_lag, max(xcorrs, key=abs)),
                              xytext=(best_lag+0.5, max(xcorrs, key=abs)*0.8),
                              fontsize=8, color=ORANGE,
                              arrowprops=dict(arrowstyle='->', color=ORANGE, lw=1.2))
    plt.tight_layout()
    plt.savefig(FIGS / 'eda_05_macro_xcorr.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Peak transmission lag found — use as input for NB04 lag selection.')
else:
    print('Insufficient time periods for cross-correlation analysis (need ≥ 8).')

In [ ]:
# --- Dual-axis time series: NPL vs macro cycle ---
if len(system_ts) >= 4:
    fig, ax1 = plt.subplots(figsize=(11, 4.5))
    ax2 = ax1.twinx()

    ax1.plot(system_ts.index, system_ts['npl_ratio'],
             color=MAROON, lw=2.2, label='NPL ratio (left)')
    ax1.fill_between(system_ts.index, system_ts['npl_ratio'], alpha=0.08, color=MAROON)
    ax1.set_ylabel('NPL Ratio', color=MAROON)
    ax1.tick_params(axis='y', labelcolor=MAROON)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))

    if 'activity_growth' in system_ts.columns:
        ax2.bar(system_ts.index, system_ts['activity_growth'],
                color=TEAL, alpha=0.35, width=25, label='Activity growth (right)')
        ax2.axhline(0, color=GRAY, lw=0.6)
        ax2.set_ylabel('Activity Growth (YoY)', color=TEAL)
        ax2.tick_params(axis='y', labelcolor=TEAL)
        ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1+lines2, labels1+labels2, loc='upper left', frameon=False, fontsize=9)
    ax1.set_title('Macro-Credit Cycle: NPL Ratio vs Economic Activity', pad=10)
    ax1.spines['top'].set_visible(False)
    plt.tight_layout()
    plt.savefig(FIGS / 'eda_06_macro_credit_cycle.png', bbox_inches='tight', dpi=150)
    plt.show()

## EDA Summary — Modeling Decisions

In [ ]:
print('=' * 60)
print('EDA SUMMARY — INPUTS FOR NB04 MODEL DESIGN')
print('=' * 60)

# Target imbalance
target_rate = df[TARGET].mean()
print(f'\n[TARGET]')
print(f'  Deterioration rate  : {target_rate:.2%}')
if target_rate < 0.20 or target_rate > 0.80:
    print('  → Imbalanced target. Use class_weight=balanced or oversample.')
else:
    print('  → Reasonably balanced. Standard logit applicable.')

# Systemic episodes
print(f'\n[SYSTEMIC RISK]')
print(f'  High-deterioration periods: {len(high_periods)} — ', end='')
print('time dummies recommended' if len(high_periods) > 0 else 'idiosyncratic pattern')

# Top features by signal
print(f'\n[TOP FEATURES by |point-biserial r|]')
for _, row in corr_df.head(5).iterrows():
    sig = '*' if row['significant'] else ' '
    print(f'  {sig} {row["feature"]:<30} r={row["point_biserial_r"]:+.4f}  [{row["signal"]}]')

# Collinearity
print(f'\n[COLLINEARITY]')
if high_corr:
    for a, b, r in high_corr:
        print(f'  DROP candidate: {a} or {b}  (r={r})')
else:
    print('  No critical pairs (|r|>0.80) — full feature set safe for logit.')

print(f'\n[MACRO TRANSMISSION]')
print('  Cross-correlation plot saved → inspect peak lag for NB04 lag specification.')

print('\nNext → 04_pd_model.ipynb')